# Preparação das bases para o dashboard EduLens

Execute este notebook depois de `01_exploracao_dados.ipynb`, que gera a base integrada `Dados/Tratados/painel_escolas_sp_2023_2025_com_ideb.csv`.

Este notebook produz duas tabelas:

- `Dados/Tratados/dashboard_infraestrutura_escolas_long.csv`: detalhe por escola e indicador, com os valores de 2023 e 2025, transição e características escolares. **O app `app.py` lê esta tabela.**
- `Dados/Tratados/dashboard_infraestrutura_municipios.csv`: agregação por município, localização, rede administrativa, categoria e indicador, útil para análises e exportações.

A cobertura anual é a média do indicador binário (0/1) entre os valores conhecidos daquele ano; ausências ficam fora do denominador. A variação é expressa em pontos percentuais. Biblioteca e sala de leitura são mantidas como indicadores distintos. Localização e rede administrativa usam 2023 como referência.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from time import perf_counter

inicio = perf_counter()

# Funciona com o notebook aberto na raiz do projeto ou em Notebooks/.
PASTA_PROJETO = Path("..") if Path.cwd().name.lower() in {"notebooks", "notebook"} else Path(".")
DADOS_TRATADOS = PASTA_PROJETO / "Dados" / "Tratados"
caminho_base = DADOS_TRATADOS / "painel_escolas_sp_2023_2025_com_ideb.csv"
if not caminho_base.exists():
    raise FileNotFoundError("Execute primeiro 01_exploracao_dados.ipynb.")

indicadores = {
    "Serviços básicos": {"Água potável":"IN_AGUA_POTAVEL", "Energia da rede pública":"IN_ENERGIA_REDE_PUBLICA"},
    "Espaços de aprendizagem": {"Biblioteca":"IN_BIBLIOTECA", "Sala de leitura":"IN_BIBLIOTECA_SALA_LEITURA", "Laboratório de ciências":"IN_LABORATORIO_CIENCIAS", "Laboratório de informática":"IN_LABORATORIO_INFORMATICA", "Quadra de esportes":"IN_QUADRA_ESPORTES"},
    "Conectividade": {"Internet":"IN_INTERNET", "Internet para alunos":"IN_INTERNET_ALUNOS", "Internet para aprendizagem":"IN_INTERNET_APRENDIZAGEM", "Banda larga":"IN_BANDA_LARGA"},
    "Equipamentos para alunos": {"Desktop":"IN_DESKTOP_ALUNO", "Computador portátil":"IN_COMP_PORTATIL_ALUNO", "Tablet":"IN_TABLET_ALUNO"},
}
contexto = ["CO_ENTIDADE", "NO_ENTIDADE_2023", "CO_MUNICIPIO_2023", "NO_MUNICIPIO_2023", "SG_UF_2023", "TP_DEPENDENCIA_2023", "TP_LOCALIZACAO_2023", "VL_OBSERVADO_2023", "VL_OBSERVADO_2025"]
flags = [f"{v}_{ano}" for grupo in indicadores.values() for v in grupo.values() for ano in (2023, 2025)]
print("Lendo o painel integrado...", flush=True)
df = pd.read_csv(caminho_base, usecols=contexto + flags, dtype={"CO_ENTIDADE":"string", "CO_MUNICIPIO_2023":"string"}, encoding="utf-8-sig")
print(f"Base carregada: {len(df):,} escolas.", flush=True)
for c in flags + ["VL_OBSERVADO_2023", "VL_OBSERVADO_2025"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

localizacao = df["TP_LOCALIZACAO_2023"].map({1:"Urbana", 2:"Rural"}).fillna("Não classificada")
rede = df["TP_DEPENDENCIA_2023"].map({1:"Federal", 2:"Estadual", 3:"Municipal"}).fillna("Não classificada")
contexto_dashboard = pd.DataFrame({
    "CO_ENTIDADE":df["CO_ENTIDADE"], "NO_ENTIDADE":df["NO_ENTIDADE_2023"],
    "CO_MUNICIPIO":df["CO_MUNICIPIO_2023"], "NO_MUNICIPIO":df["NO_MUNICIPIO_2023"],
    "SG_UF":df["SG_UF_2023"], "REDE_ADMINISTRATIVA_2023":rede,
    "LOCALIZACAO_2023":localizacao, "IDEB_2023":df["VL_OBSERVADO_2023"],
    "IDEB_2025":df["VL_OBSERVADO_2025"]})
contexto_dashboard["DELTA_IDEB"] = contexto_dashboard["IDEB_2025"] - contexto_dashboard["IDEB_2023"]

print("Preparando uma linha por escola e indicador...", flush=True)
partes = []
for categoria, grupo in indicadores.items():
    for nome, codigo in grupo.items():
        p = contexto_dashboard.copy()
        p["CATEGORIA"], p["INDICADOR"], p["INDICADOR_CODIGO"] = categoria, nome, codigo
        p["VALOR_2023"], p["VALOR_2025"] = df[f"{codigo}_2023"], df[f"{codigo}_2025"]
        a, b = p["VALOR_2023"], p["VALOR_2025"]
        p["TRANSICAO"] = np.select(
            [a.isna() | b.isna(), a.eq(0)&b.eq(0), a.eq(0)&b.eq(1), a.eq(1)&b.eq(0), a.eq(1)&b.eq(1)],
            ["Sem comparação completa", "Permaneceu sem recurso", "Ganhou recurso", "Perdeu registro do recurso", "Manteve recurso"],
            default="Valor inválido")
        partes.append(p)

dashboard_escolas = pd.concat(partes, ignore_index=True)
print(f"Tabela longa pronta: {len(dashboard_escolas):,} linhas.", flush=True)

# Pré-calcula contagens de transição em colunas numéricas para agregar sem funções Python por grupo.
transicoes = {
    "N_GANHOU_RECURSO": "Ganhou recurso",
    "N_PERDEU_REGISTRO": "Perdeu registro do recurso",
    "N_PERSISTENTE_SEM_RECURSO": "Permaneceu sem recurso",
    "N_MANTEVE_RECURSO": "Manteve recurso",
    "N_SEM_COMPARACAO": "Sem comparação completa",
}
for coluna, rotulo in transicoes.items():
    dashboard_escolas[coluna] = dashboard_escolas["TRANSICAO"].eq(rotulo).astype("int8")

print("Agregando por município, localização e rede...", flush=True)
chaves = ["CO_MUNICIPIO", "NO_MUNICIPIO", "LOCALIZACAO_2023", "REDE_ADMINISTRATIVA_2023", "CATEGORIA", "INDICADOR", "INDICADOR_CODIGO"]
agrupado = dashboard_escolas.groupby(chaves, dropna=False, sort=False)
metricas = {
    "N_ESCOLAS": ("CO_ENTIDADE", "count"),
    "N_COM_DADO_2023": ("VALOR_2023", "count"),
    "N_COM_DADO_2025": ("VALOR_2025", "count"),
    "MEDIA_2023": ("VALOR_2023", "mean"),
    "MEDIA_2025": ("VALOR_2025", "mean"),
}
metricas.update({nome: (nome, "sum") for nome in transicoes})
dashboard_municipios = agrupado.agg(**metricas).reset_index()
dashboard_municipios["COBERTURA_2023_PCT"] = dashboard_municipios["MEDIA_2023"] * 100
dashboard_municipios["COBERTURA_2025_PCT"] = dashboard_municipios["MEDIA_2025"] * 100
dashboard_municipios["VARIACAO_COBERTURA_PP"] = dashboard_municipios["COBERTURA_2025_PCT"] - dashboard_municipios["COBERTURA_2023_PCT"]
dashboard_municipios = dashboard_municipios.drop(columns=["MEDIA_2023", "MEDIA_2025"])

print("Salvando arquivos para o dashboard...", flush=True)
arquivo_escolas = DADOS_TRATADOS / "dashboard_infraestrutura_escolas_long.csv"
arquivo_municipios = DADOS_TRATADOS / "dashboard_infraestrutura_municipios.csv"
dashboard_escolas.to_csv(arquivo_escolas, index=False, encoding="utf-8-sig")
dashboard_municipios.to_csv(arquivo_municipios, index=False, encoding="utf-8-sig")
print("Indicadores:", sum(len(v) for v in indicadores.values()))
print("Escolas:", df["CO_ENTIDADE"].nunique())
print("Linhas detalhadas:", len(dashboard_escolas))
print("Linhas agregadas:", len(dashboard_municipios))
print("CSV detalhado:", arquivo_escolas.relative_to(PASTA_PROJETO))
print("CSV agregado:", arquivo_municipios.relative_to(PASTA_PROJETO))
print(f"Tempo total: {perf_counter()-inicio:.1f} s.", flush=True)
print("\nPrévia:")
print(dashboard_municipios[["NO_MUNICIPIO","LOCALIZACAO_2023","CATEGORIA","INDICADOR","N_ESCOLAS","COBERTURA_2023_PCT","COBERTURA_2025_PCT","VARIACAO_COBERTURA_PP","N_GANHOU_RECURSO","N_PERDEU_REGISTRO"]].head(6).round(2).to_string(index=False))
